
# `gu_toolkit` layout debugging notebook

This notebook is aimed at **layout regressions, resize bugs, sidebar issues, modal/overlay bugs, and widget-tree inspection** in `gu_toolkit`.

It combines:

- **fast automated checks** against the toolkit’s layout contracts,
- **live interactive labs** for `Figure`, `PlotlyPane`, and `FloatSlider`,
- **visual stress cases** for narrow hosts, wrapped sidebars, and hidden-to-visible containers,
- a **scratch reproducer** you can edit to mirror any failing notebook cell from your project.

The notebook is intentionally a little introspection-heavy, so it can help you answer both of these questions:

1. **What does the runtime widget tree actually look like?**
2. **Which layout trait or container change is causing the visual failure?**


In [1]:
!pip -q install plotly pandas scipy numpy sympy anywidget ipywidgets
# Setup: Import toolkit
import conftest # helps find the src
from gu_toolkit import *


In [2]:
import inspect
import logging
import textwrap

import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import HTML, Markdown, clear_output, display

from gu_toolkit.Figure import Figure
from gu_toolkit.PlotlyPane import PlotlyPane, PlotlyPaneStyle
from gu_toolkit.Slider import FloatSlider
from gu_toolkit.figure_info import InfoPanelManager
from gu_toolkit.figure_layout import FigureLayout, OneShotOutput
from gu_toolkit.figure_parameters import ParameterManager

logging.basicConfig(level=logging.INFO, force=True)
for name in ["gu_toolkit.Figure", "gu_toolkit.layout", "gu_toolkit.layout.plotly_pane", "gu_toolkit.layout.plotly_driver"]:
    logging.getLogger(name).setLevel(logging.DEBUG)



## 1. Confirm the notebook is using the code you expect

Before debugging layout, verify the imported classes are resolving to the **local toolkit source** you meant to test.


In [3]:
targets = {
    "Figure": Figure,
    "FigureLayout": FigureLayout,
    "PlotlyPane": PlotlyPane,
    "FloatSlider": FloatSlider,
    "ParameterManager": ParameterManager,
    "InfoPanelManager": InfoPanelManager,
    "OneShotOutput": OneShotOutput,
}

runtime_sources = pd.DataFrame(
    [
        {
            "name": name,
            "source_file": inspect.getsourcefile(obj),
            "signature": str(inspect.signature(obj)) if callable(obj) else "",
        }
        for name, obj in targets.items()
    ]
)

runtime_sources


,name,source_file,signature
0,Figure,C:\Users\guraltsev\Documents\notebooks\gu_tool...,"(*, title: 'str' = '', sampling_points: 'int' ..."
1,FigureLayout,C:\Users\guraltsev\Documents\notebooks\gu_tool...,(title: 'str' = '') -> 'None'
2,PlotlyPane,C:\Users\guraltsev\Documents\notebooks\gu_tool...,"(figw: 'W.Widget', *, style: 'PlotlyPaneStyle'..."
3,FloatSlider,C:\Users\guraltsev\Documents\notebooks\gu_tool...,"(value: float = 0.0, min: float = 0.0, max: fl..."
4,ParameterManager,C:\Users\guraltsev\Documents\notebooks\gu_tool...,"(render_callback: 'Callable[[str, ParamEvent],..."
5,InfoPanelManager,C:\Users\guraltsev\Documents\notebooks\gu_tool...,(layout_box: 'widgets.Box') -> 'None'
6,OneShotOutput,C:\Users\guraltsev\Documents\notebooks\gu_tool...,() -> 'None'



## 2. Debug helpers

These helpers keep the rest of the notebook compact:

- `show_tree(widget)` dumps the widget hierarchy and key layout traits.
- `figure_diag(fig)`, `pane_diag(pane)`, and `slider_diag(slider)` summarize internal state.
- `framed(...)` puts any widget into a visually obvious host box so width/height issues are easier to reproduce.


In [4]:

LAYOUT_KEYS = [
    "display",
    "width",
    "height",
    "min_width",
    "min_height",
    "max_width",
    "max_height",
    "flex",
    "flex_flow",
    "align_items",
    "justify_content",
    "overflow",
    "padding",
    "margin",
    "border",
    "border_radius",
    "position",
    "top",
    "left",
    "gap",
    "box_sizing",
]

def layout_snapshot(widget, keys=LAYOUT_KEYS):
    layout = getattr(widget, "layout", None)
    if layout is None:
        return {}
    out = {}
    for key in keys:
        if hasattr(layout, key):
            value = getattr(layout, key)
            if value not in (None, "", (), []):
                out[key] = value
    return out

def widget_tree(widget, path="root", depth=0, max_depth=5):
    rows = []
    rows.append(
        {
            "path": path,
            "depth": depth,
            "type": type(widget).__name__,
            "n_children": len(getattr(widget, "children", ()) or ()),
            "dom_classes": tuple(getattr(widget, "_dom_classes", ())),
            "layout": layout_snapshot(widget),
        }
    )
    if depth >= max_depth:
        return rows

    for i, child in enumerate(getattr(widget, "children", ()) or ()):
        if isinstance(child, widgets.Widget):
            child_path = f"{path}/{type(child).__name__}[{i}]"
            rows.extend(widget_tree(child, child_path, depth + 1, max_depth=max_depth))
        else:
            rows.append(
                {
                    "path": f"{path}/<{type(child).__name__}>[{i}]",
                    "depth": depth + 1,
                    "type": type(child).__name__,
                    "n_children": None,
                    "dom_classes": (),
                    "layout": {},
                }
            )
    return rows

def show_tree(widget, max_depth=4):
    df = pd.DataFrame(widget_tree(widget, max_depth=max_depth))
    cols = ["path", "depth", "type", "n_children", "dom_classes", "layout"]
    return df[cols]

def _check_summary(checks):
    failing = [name for name, ok in checks.items() if not ok]
    return (not failing), ("all good" if not failing else "failed: " + ", ".join(failing))

def active_view_host(fig):
    return fig._layout._view_pages[fig.views.current_id].host_box

def figure_diag(fig):
    snap = {
        "title": fig.title,
        "plots": len(fig.plots),
        "params": len(fig.parameters),
        "info_outputs": len(fig.info_output),
        "full_width": fig._layout.full_width_checkbox.value,
        "content_layout_mode": fig._layout.content_layout_mode,
        "content_flex_flow": fig._layout.content_wrapper.layout.flex_flow,
        "view_stage_height": fig._layout.view_stage.layout.height,
        "view_stage_min_width": fig._layout.view_stage.layout.min_width,
        "sidebar_display": fig._layout.sidebar_container.layout.display,
        "sidebar_width": fig._layout.sidebar_container.layout.width,
        "sidebar_min_width": fig._layout.sidebar_container.layout.min_width,
        "sidebar_max_width": fig._layout.sidebar_container.layout.max_width,
        "params_display": fig._layout.params_box.layout.display,
        "info_display": fig._layout.info_box.layout.display,
        "print_min_height": fig._layout.print_output.layout.min_height,
        "active_view_host_display": active_view_host(fig).layout.display,
        "pane_wrap_width": fig.pane._wrap.layout.width,
        "pane_wrap_height": fig.pane._wrap.layout.height,
        "pane_host_width": fig.pane._host.layout.width,
        "pane_host_height": fig.pane._host.layout.height,
        "driver_autorange_mode": fig.pane.driver.autorange_mode,
        "driver_defer_reveal": fig.pane.driver.defer_reveal,
        "driver_debounce_ms": fig.pane.driver.debounce_ms,
        "driver_min_delta_px": fig.pane.driver.min_delta_px,
    }
    for key, value in fig.pane.geometry.as_dict().items():
        snap[f"pane_{key}"] = value
    return snap

def pane_diag(pane):
    return pane.debug_snapshot()

def slider_diag(slider):
    if slider.settings_modal in slider.children:
        modal_parent = "slider"
    else:
        modal_parent = "host/other"
    return {
        "value": slider.value,
        "slider_min": slider.slider.min,
        "slider_max": slider.slider.max,
        "slider_step": slider.slider.step,
        "continuous_update": slider.slider.continuous_update,
        "modal_parent": modal_parent,
        "modal_display": slider.settings_modal.layout.display,
        "panel_display": slider.settings_panel.layout.display,
        "modal_host_is_none": slider._modal_host is None,
        "slider_children": len(slider.children),
    }

def framed(widget, *, label=None, width="100%", height=None, overflow="visible",
           border="1px dashed #94a3b8", padding="6px"):
    layout = widgets.Layout(
        width=width,
        overflow=overflow,
        border=border,
        padding=padding,
    )
    if height is not None:
        layout.height = height
    box = widgets.Box([widget], layout=layout)
    if label is None:
        return box
    return widgets.VBox(
        [
            widgets.HTML(f"<b>{label}</b>"),
            box,
        ],
        layout=widgets.Layout(width=width),
    )

def make_demo_figure(
    title="Demo figure",
    *,
    info=True,
    params=("a", "b"),
    traces=2,
    plot_height="320px",
):
    symbol_map = {
        "a": a,
        "b": b,
        "c": c,
        "d": d,
    }

    defaults = {
        "a": dict(min=-2, max=2, value=1.0, step=0.1),
        "b": dict(min=-pi, max=pi, value=0.0, step=0.1),
        "c": dict(min=-3, max=3, value=0.5, step=0.1),
        "d": dict(min=-4, max=4, value=-1.0, step=0.1),
    }

    fig = Figure(default_x_range=(-6, 6), default_y_range=(-3, 3))
    fig._layout.view_stage.layout.height = plot_height
    fig.title = title

    fig.plot(sin(x), x, id="sin", color="#1f77b4")
    if traces >= 2:
        fig.plot(symbol_map["a"] * cos(x + symbol_map["b"]), x, id="a_cos", color="#d62728", dash="dash")
    if traces >= 3:
        fig.plot(0.5 * sin(2 * x), x, id="fast", opacity=0.45, dash="dot")
    if traces >= 4:
        fig.plot(0.35 * cos(3 * x), x, id="faster", opacity=0.35)

    for name in params:
        sym = symbol_map[name] if isinstance(name, str) else name
        cfg = defaults.get(getattr(sym, "name", str(sym)), dict(min=-2, max=2, value=0.0, step=0.1))
        fig.parameter(sym, **cfg)

    if info:
        out = fig.info_manager.get_output("summary", height="90px")
        with out:
            print("Info panel is live.")
            print("Resize the host, toggle full width, and inspect the widget tree.")

    with fig:
        print("Output-capture area is active inside `with fig:` blocks.")

    return fig



## 3. Fast automated checks

These checks verify the main layout contracts visible in the source:

- empty `Figure` state,
- sidebar activation for parameters and info outputs,
- full-width toggle behavior,
- parameter reuse behavior,
- `PlotlyPane` wrapper/host contract,
- `FloatSlider` modal reparenting.


In [5]:

def run_layout_checks():
    rows = []

    def add(name, fn):
        try:
            result = fn()
            rows.append({"check": name, **result})
        except Exception as exc:
            rows.append(
                {
                    "check": name,
                    "passed": False,
                    "detail": f"{type(exc).__name__}: {exc}",
                }
            )

    def t_empty_figure():
        fig = Figure()
        checks = {
            "sidebar_hidden": fig._layout.sidebar_container.layout.display == "none",
            "row_wrap_default": fig._layout.content_wrapper.layout.flex_flow == "row wrap",
            "plot_height_default": fig._layout.view_stage.layout.height == "60vh",
            "pane_attached": active_view_host(fig).children == (fig.pane.widget,),
            "pane_wrap_100pct_height": fig.pane._wrap.layout.height == "100%",
            "pane_host_100pct_height": fig.pane._host.layout.height == "100%",
        }
        passed, detail = _check_summary(checks)
        return {"passed": passed, "detail": detail, **checks}

    def t_params_activate_sidebar():
        fig = Figure()
        ref = fig.parameter(a, min=-2, max=2, value=0.25, step=0.1)
        control = fig.parameters.widgets()[0]
        checks = {
            "has_params": fig.parameters.has_params,
            "sidebar_visible": fig._layout.sidebar_container.layout.display == "flex",
            "params_box_visible": fig._layout.params_box.layout.display == "flex",
            "is_float_slider": isinstance(control, FloatSlider),
            "ref_widget_matches_control": getattr(ref, "widget", None) is control,
            "modal_host_bound": getattr(control, "_modal_host", None) is fig._layout.root_widget,
            "modal_reparented": (control.settings_modal in fig._layout.root_widget.children)
                                and (control.settings_modal not in control.children),
        }
        passed, detail = _check_summary(checks)
        return {"passed": passed, "detail": detail, **checks}

    def t_info_activate_sidebar():
        fig = Figure()
        out = fig.info_manager.get_output("summary", height="90px")
        checks = {
            "has_info": fig.info_manager.has_info,
            "sidebar_visible": fig._layout.sidebar_container.layout.display == "flex",
            "info_box_visible": fig._layout.info_box.layout.display == "flex",
            "params_box_hidden": fig._layout.params_box.layout.display == "none",
            "output_registered": out in fig._layout.info_box.children,
            "output_id_kept": getattr(out, "id", None) == "summary",
            "layout_updated": out.layout.height == "90px",
        }
        passed, detail = _check_summary(checks)
        return {"passed": passed, "detail": detail, **checks}

    def t_info_output_reuse():
        fig = Figure()
        out1 = fig.info_manager.get_output("summary", height="80px")
        out2 = fig.info_manager.get_output("summary", height="120px")
        checks = {
            "same_object": out1 is out2,
            "height_updated": out2.layout.height == "120px",
            "single_output": len(fig.info_output) == 1,
        }
        passed, detail = _check_summary(checks)
        return {"passed": passed, "detail": detail, **checks}

    def t_full_width_toggle():
        fig = Figure()
        fig.parameter(a, min=-2, max=2, value=1.0, step=0.1)
        fig.info_manager.get_output("summary", height="80px")

        fig._layout.full_width_checkbox.value = True
        checks_on = {
            "column_mode": fig._layout.content_wrapper.layout.flex_flow == "column",
            "sidebar_width_100pct": fig._layout.sidebar_container.layout.width == "100%",
            "sidebar_padding_zero": fig._layout.sidebar_container.layout.padding == "0px",
        }

        fig._layout.full_width_checkbox.value = False
        checks_off = {
            "wrap_mode_restored": fig._layout.content_wrapper.layout.flex_flow == "row wrap",
            "sidebar_width_auto": fig._layout.sidebar_container.layout.width == "auto",
            "sidebar_max_width_restored": fig._layout.sidebar_container.layout.max_width == "400px",
        }

        checks = {**checks_on, **checks_off}
        passed, detail = _check_summary(checks)
        return {"passed": passed, "detail": detail, **checks}

    def t_parameter_reuse():
        fig = Figure()
        ref1 = fig.parameter(a, min=-2, max=2, value=0.5, step=0.1)
        control1 = fig.parameters.widgets()[0]
        ref2 = fig.parameter(a, min=-5, max=5, value=1.25, step=0.25)
        control2 = fig.parameters.widgets()[0]

        checks = {
            "same_ref_object": ref1 is ref2,
            "same_control_object": control1 is control2,
            "single_control": len(fig.parameters.widgets()) == 1,
            "value_updated": abs(ref1.value - 1.25) < 1e-12,
        }

        for attr, expected in [("min", -5.0), ("max", 5.0), ("step", 0.25)]:
            if hasattr(ref1, attr):
                checks[f"{attr}_updated"] = abs(getattr(ref1, attr) - expected) < 1e-12

        passed, detail = _check_summary(checks)
        return {"passed": passed, "detail": detail, **checks}

    def t_plotly_pane_contract():
        figw = go.FigureWidget(data=[go.Scatter(y=[1, 3, 2])])
        pane = PlotlyPane(
            figw,
            autorange_mode="once",
            defer_reveal=True,
            debounce_ms=70,
            min_delta_px=3,
        )
        checks = {
            "wrap_has_one_child": len(pane._wrap.children) == 1,
            "host_has_plot_slot_and_driver": len(pane._host.children) == 2,
            "wrap_width_100pct": pane._wrap.layout.width == "100%",
            "wrap_height_100pct": pane._wrap.layout.height == "100%",
            "host_width_100pct": pane._host.layout.width == "100%",
            "host_height_100pct": pane._host.layout.height == "100%",
            "host_flex_column": pane._host.layout.flex_flow == "column",
            "driver_mode_kept": pane.driver.autorange_mode == "once",
            "driver_defer_reveal_kept": pane.driver.defer_reveal is True,
        }
        passed, detail = _check_summary(checks)
        return {"passed": passed, "detail": detail, **checks}

    def t_slider_modal_reparent():
        slider = FloatSlider(value=0.5, min=-1, max=1, step=0.1, description="$a$")
        host = widgets.Box()

        checks = {
            "starts_local": slider.settings_modal in slider.children,
        }

        slider.set_modal_host(host)
        checks["moves_to_host"] = (slider.settings_modal in host.children) and (slider.settings_modal not in slider.children)

        slider.set_modal_host(None)
        checks["moves_back_local"] = (slider.settings_modal in slider.children) and (slider.settings_modal not in host.children)

        passed, detail = _check_summary(checks)
        return {"passed": passed, "detail": detail, **checks}

    add("Empty Figure contracts", t_empty_figure)
    add("Parameter sidebar activation", t_params_activate_sidebar)
    add("Info sidebar activation", t_info_activate_sidebar)
    add("Info output reuse", t_info_output_reuse)
    add("Full-width toggle contract", t_full_width_toggle)
    add("Parameter reuse / update-in-place", t_parameter_reuse)
    add("PlotlyPane host/wrapper contract", t_plotly_pane_contract)
    add("FloatSlider modal reparenting", t_slider_modal_reparent)

    return pd.DataFrame(rows)

layout_checks = run_layout_checks()
layout_checks.insert(0, "status", layout_checks["passed"].map({True: "✅ PASS", False: "❌ FAIL"}))
layout_checks


INFO:gu_toolkit.layout.figure:{'ts': 1773841466.871738, 'event': 'relayout_debouncer_created', 'source': 'Figure', 'phase': 'completed', 'figure_id': 'figure-0001-d251fa', 'seq': 1}
DEBUG:gu_toolkit.layout.figure:{'ts': 1773841467.713246, 'event': 'view_runtime_created', 'source': 'Figure', 'phase': 'completed', 'figure_id': 'figure-0001-d251fa', 'view_id': 'main', 'pane_id': 'pane-0002-3d90f5', 'default_x_range': [-4.0, 4.0], 'default_y_range': [-3.0, 3.0], 'active_view_id': 'main', 'seq': 2}
DEBUG:gu_toolkit.layout.figure:{'ts': 1773841467.714665, 'event': 'view_selector_refreshed', 'source': 'FigureLayout', 'phase': 'completed', 'figure_id': 'figure-0001-d251fa', 'options': ['main'], 'selector_display': 'none', 'active_view_id': 'main', 'seq': 3}
DEBUG:gu_toolkit.layout.figure:{'ts': 1773841467.714972, 'event': 'view_order_changed', 'source': 'FigureLayout', 'phase': 'completed', 'figure_id': 'figure-0001-d251fa', 'ordered_view_ids': ['main'], 'active_view_id': 'main', 'seq': 4}
DEB

,status,check,passed,detail,sidebar_hidden,row_wrap_default,plot_height_default,pane_attached,pane_wrap_100pct_height,pane_host_100pct_height,...,wrap_width_100pct,wrap_height_100pct,host_width_100pct,host_height_100pct,host_flex_column,driver_mode_kept,driver_defer_reveal_kept,starts_local,moves_to_host,moves_back_local
0,✅ PASS,Empty Figure contracts,True,all good,True,True,True,True,True,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,✅ PASS,Parameter sidebar activation,True,all good,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,❌ FAIL,Info sidebar activation,False,"failed: sidebar_visible, info_box_visible",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,✅ PASS,Info output reuse,True,all good,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,✅ PASS,Full-width toggle contract,True,all good,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,✅ PASS,Parameter reuse / update-in-place,True,all good,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,✅ PASS,PlotlyPane host/wrapper contract,True,all good,NaN,NaN,NaN,NaN,NaN,NaN,...,True,True,True,True,True,True,True,NaN,NaN,NaN
7,✅ PASS,FloatSlider modal reparenting,True,all good,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,True,True


In [6]:

failed_layout_checks = layout_checks.loc[~layout_checks["passed"], ["check", "detail"]]
failed_layout_checks if len(failed_layout_checks) else "All automated layout checks passed."


,check,detail
2,Info sidebar activation,"failed: sidebar_visible, info_box_visible"



## 4. Live `Figure` layout lab

Use this lab to reproduce **sidebar wrapping**, **column/full-width mode**, **host-width squeeze**, and **manual reflow** behavior on a real `Figure`.


In [7]:

def build_figure_lab():
    fig = make_demo_figure("Figure layout lab", info=True, params=("a", "b"), traces=3, plot_height="320px")

    host = widgets.Box(
        [fig._layout.root_widget],
        layout=widgets.Layout(width="1000px", border="1px dashed #94a3b8", padding="6px", overflow="auto"),
    )

    layout_mode = widgets.ToggleButtons(
        options=[("Wrapped sidebar", "wrapped"), ("Column / full width", "column")],
        value="wrapped",
        description="Mode:",
    )
    host_width = widgets.Dropdown(
        options=["1000px", "760px", "460px", "340px", "100%"],
        value="1000px",
        description="Host width:",
    )
    plot_height = widgets.Dropdown(
        options=["220px", "320px", "420px", "60vh"],
        value="320px",
        description="Plot height:",
    )
    force_reflow = widgets.Button(description="Force reflow")

    diag = widgets.Output()

    def refresh(*_):
        fig._layout.full_width_checkbox.value = layout_mode.value == "column"
        fig._layout.view_stage.layout.height = plot_height.value
        host.layout.width = host_width.value
        fig.pane.reflow()
        with diag:
            clear_output(wait=True)
            display(pd.DataFrame([figure_diag(fig)]))

    layout_mode.observe(refresh, names="value")
    host_width.observe(refresh, names="value")
    plot_height.observe(refresh, names="value")
    force_reflow.on_click(refresh)

    refresh()

    controls = widgets.HBox(
        [layout_mode, host_width, plot_height, force_reflow],
        layout=widgets.Layout(flex_flow="row wrap", gap="8px"),
    )

    ui = widgets.VBox([controls, host, diag], layout=widgets.Layout(width="100%"))
    return {"ui": ui, "figure": fig, "host": host}

figure_lab = build_figure_lab()
display(figure_lab["ui"])


INFO:gu_toolkit.layout.figure:{'ts': 1773841468.142327, 'event': 'relayout_debouncer_created', 'source': 'Figure', 'phase': 'completed', 'figure_id': 'figure-0014-341bff', 'seq': 1}
DEBUG:gu_toolkit.layout.figure:{'ts': 1773841468.195444, 'event': 'view_runtime_created', 'source': 'Figure', 'phase': 'completed', 'figure_id': 'figure-0014-341bff', 'view_id': 'main', 'pane_id': 'pane-0015-ad2aa7', 'default_x_range': [-6.0, 6.0], 'default_y_range': [-3.0, 3.0], 'active_view_id': 'main', 'seq': 2}
DEBUG:gu_toolkit.layout.figure:{'ts': 1773841468.198584, 'event': 'view_selector_refreshed', 'source': 'FigureLayout', 'phase': 'completed', 'figure_id': 'figure-0014-341bff', 'options': ['main'], 'selector_display': 'none', 'active_view_id': 'main', 'seq': 3}
DEBUG:gu_toolkit.layout.figure:{'ts': 1773841468.199454, 'event': 'view_order_changed', 'source': 'FigureLayout', 'phase': 'completed', 'figure_id': 'figure-0014-341bff', 'ordered_view_ids': ['main'], 'active_view_id': 'main', 'seq': 4}
DEB


Inspect the live widget tree behind the lab figure:


In [8]:

show_tree(smartfigure_lab["figure"]._layout.root_widget, max_depth=4)


NameError: name 'smartfigure_lab' is not defined


## 5. Narrow-host wrap matrix

This is a **visual regression matrix** for one of the most common layout problems: the figure works at desktop width and breaks when the notebook pane or parent container gets narrow.

The toolkit’s layout uses a wrapping content row, so the sidebar should eventually drop below the plot instead of crushing it.


In [ ]:

def make_wrap_case(width, title):
    fig = make_demo_figure(title, info=True, params=("a", "b"), traces=2, plot_height="260px")
    box = widgets.Box(
        [fig._layout.root_widget],
        layout=widgets.Layout(width=width, border="1px dashed #94a3b8", padding="6px", overflow="auto"),
    )
    card = widgets.VBox(
        [
            widgets.HTML(f"<b>{title}</b><br><code>host width = {width}</code>"),
            box,
        ],
        layout=widgets.Layout(width="100%", gap="6px"),
    )
    return card, fig

wrap_cards = []
wrap_figs = []

for width in ["1100px", "760px", "460px"]:
    card, fig = make_wrap_case(width, f"Figure at {width}")
    wrap_cards.append(card)
    wrap_figs.append(fig)

display(
    widgets.VBox(
        wrap_cards,
        layout=widgets.Layout(width="100%", gap="14px"),
    )
)



## 6. Direct `PlotlyPane` lab

This strips away `Figure` and exercises the **raw resize container** itself. Use it when the bug looks more like:

- a Plotly chart failing to resize after the notebook pane changes width,
- a clipped plot inside a scrolling parent,
- a chart that needs explicit `reflow()` after a layout mutation.


In [ ]:

def build_plotly_pane_lab():
    x_values = np.linspace(0, 2 * np.pi, 500)
    figw = go.FigureWidget(
        data=[
            go.Scatter(x=x_values, y=np.sin(x_values), name="sin(x)"),
            go.Scatter(x=x_values, y=0.35 * np.cos(3 * x_values), name="0.35 cos(3x)"),
        ]
    )
    figw.update_layout(title="Direct PlotlyPane lab", template="plotly_white")

    pane = PlotlyPane(
        figw,
        style=PlotlyPaneStyle(padding_px=8, border="1px solid rgba(15,23,42,0.12)", border_radius_px=10),
        autorange_mode="once",
        defer_reveal=True,
        debug_js=False,
    )

    inner = widgets.Box(
        [pane.widget],
        layout=widgets.Layout(width="100%", height="320px", border="1px dashed #94a3b8", padding="4px"),
    )
    clip = widgets.Box(
        [inner],
        layout=widgets.Layout(width="760px", height="360px", overflow="hidden", border="2px solid #f59e0b", padding="4px"),
    )

    clip_width = widgets.Dropdown(
        options=["1000px", "760px", "460px", "320px", "100%"],
        value="760px",
        description="Clip width:",
    )
    inner_height = widgets.Dropdown(
        options=["220px", "320px", "420px"],
        value="320px",
        description="Plot height:",
    )
    overflow = widgets.Dropdown(
        options=["hidden", "auto", "visible"],
        value="hidden",
        description="Overflow:",
    )
    autorange = widgets.Dropdown(
        options=["none", "once", "always"],
        value="once",
        description="Autorange:",
    )
    force_reflow = widgets.Button(description="Force reflow")

    diag = widgets.Output()

    def refresh(*_):
        clip.layout.width = clip_width.value
        inner.layout.height = inner_height.value
        clip.layout.overflow = overflow.value
        pane.driver.autorange_mode = autorange.value
        pane.reflow()
        with diag:
            clear_output(wait=True)
            display(pd.DataFrame([pane_diag(pane)]))

    clip_width.observe(refresh, names="value")
    inner_height.observe(refresh, names="value")
    overflow.observe(refresh, names="value")
    autorange.observe(refresh, names="value")
    force_reflow.on_click(refresh)

    refresh()

    controls = widgets.HBox(
        [clip_width, inner_height, overflow, autorange, force_reflow],
        layout=widgets.Layout(flex_flow="row wrap", gap="8px"),
    )
    ui = widgets.VBox([controls, clip, diag], layout=widgets.Layout(width="100%"))
    return {"ui": ui, "pane": pane, "clip": clip, "inner": inner}

plotly_pane_lab = build_plotly_pane_lab()
display(plotly_pane_lab["ui"])



## 7. Hidden-to-visible container test

This is a classic resize edge case. The plot is mounted inside an `Accordion`, starts hidden, and should now recover automatically when the panel becomes measurable. Manual `reflow()` is still available as an explicit override and debugging tool.


In [ ]:

def build_accordion_lab():
    x_values = np.linspace(-6, 6, 400)
    figw = go.FigureWidget(data=[go.Scatter(x=x_values, y=np.sinc(x_values / np.pi))])
    figw.update_layout(title="Accordion-hosted PlotlyPane", template="plotly_white")

    pane = PlotlyPane(figw, autorange_mode="once", defer_reveal=True)
    inner = widgets.Box([pane.widget], layout=widgets.Layout(width="100%", height="280px"))
    acc = widgets.Accordion(children=[inner], selected_index=None)
    acc.set_title(0, "Open me, then reflow")

    open_btn = widgets.Button(description="Open + reflow")
    close_btn = widgets.Button(description="Close")
    manual_reflow = widgets.Button(description="Manual reflow")
    diag = widgets.Output()

    def refresh_status():
        with diag:
            clear_output(wait=True)
            display(
                pd.DataFrame(
                    [
                        {
                            "selected_index": acc.selected_index,
                            **pane_diag(pane),
                        }
                    ]
                )
            )

    def on_open(_):
        acc.selected_index = 0
        pane.reflow()
        refresh_status()

    def on_close(_):
        acc.selected_index = None
        refresh_status()

    def on_reflow(_):
        pane.reflow()
        refresh_status()

    open_btn.on_click(on_open)
    close_btn.on_click(on_close)
    manual_reflow.on_click(on_reflow)

    refresh_status()

    controls = widgets.HBox([open_btn, close_btn, manual_reflow], layout=widgets.Layout(gap="8px"))
    ui = widgets.VBox([controls, acc, diag], layout=widgets.Layout(width="100%"))
    return {"ui": ui, "pane": pane, "accordion": acc}

accordion_lab = build_accordion_lab()
display(accordion_lab["ui"])



## 8. Missing-height vs explicit-height comparison

The `PlotlyPane` source explicitly warns that it needs a **real computed height** somewhere up the ancestor chain. This side-by-side comparison makes that failure mode easy to spot.


In [ ]:

def make_missing_height_comparison():
    xs = np.linspace(0, 10, 300)

    bad_figw = go.FigureWidget(data=[go.Scatter(x=xs, y=np.sin(xs))])
    bad_figw.update_layout(title="No explicit height upstream", template="plotly_white")
    bad_pane = PlotlyPane(bad_figw, autorange_mode="once", defer_reveal=True)
    bad_host = widgets.Box(
        [bad_pane.widget],
        layout=widgets.Layout(width="100%", border="2px dashed #ef4444", padding="6px"),
    )

    good_figw = go.FigureWidget(data=[go.Scatter(x=xs, y=np.sin(xs))])
    good_figw.update_layout(title="Explicit height upstream", template="plotly_white")
    good_pane = PlotlyPane(good_figw, autorange_mode="once", defer_reveal=True)
    good_host = widgets.Box(
        [good_pane.widget],
        layout=widgets.Layout(width="100%", height="280px", border="2px dashed #22c55e", padding="6px"),
    )

    bad_card = widgets.VBox(
        [
            widgets.HTML("<b>Expected trouble</b><br><small>No height on the parent host.</small>"),
            bad_host,
        ],
        layout=widgets.Layout(width="49%"),
    )
    good_card = widgets.VBox(
        [
            widgets.HTML("<b>Expected good behavior</b><br><small>The parent host has <code>height='280px'</code>.</small>"),
            good_host,
        ],
        layout=widgets.Layout(width="49%"),
    )

    return widgets.HBox(
        [bad_card, good_card],
        layout=widgets.Layout(width="100%", align_items="flex-start", flex_flow="row wrap", gap="12px"),
    )

display(make_missing_height_comparison())



## 9. `FloatSlider` modal-host lab

`FloatSlider` can keep its settings modal local, or it can be **reparented to a host container**. This matters for z-order, containment, and clipping bugs.


In [ ]:

def build_slider_modal_lab():
    card = widgets.Box(
        layout=widgets.Layout(width="440px", height="240px", border="1px dashed #94a3b8", padding="8px", overflow="visible")
    )
    slider = FloatSlider(value=0.5, min=-2, max=2, step=0.1, description="$a$")
    content = widgets.VBox(
        [
            widgets.HTML("<b>Slider modal lab</b><br><small>Switch between global and hosted modal placement.</small>"),
            slider,
        ],
        layout=widgets.Layout(width="100%"),
    )
    card.children = (content,)

    modal_mode = widgets.ToggleButtons(
        options=[("Global modal", "local"), ("Hosted to card", "hosted")],
        value="local",
        description="Mode:",
    )
    card_width = widgets.Dropdown(
        options=["440px", "320px", "260px"],
        value="440px",
        description="Card width:",
    )
    open_btn = widgets.Button(description="Open settings")
    close_btn = widgets.Button(description="Close settings")

    diag = widgets.Output()

    def sync_mode(*_):
        if modal_mode.value == "hosted":
            slider.set_modal_host(card)
        else:
            slider.set_modal_host(None)

        card.layout.width = card_width.value
        with diag:
            clear_output(wait=True)
            display(pd.DataFrame([slider_diag(slider)]))
            display(pd.DataFrame([{"card_children": len(card.children), "card_width": card.layout.width}]))

    def open_modal(_):
        if slider.settings_modal.layout.display != "flex":
            slider._toggle_settings(None)
        sync_mode()

    def close_modal(_):
        if slider.settings_modal.layout.display == "flex":
            slider._toggle_settings(None)
        sync_mode()

    modal_mode.observe(sync_mode, names="value")
    card_width.observe(sync_mode, names="value")
    open_btn.on_click(open_modal)
    close_btn.on_click(close_modal)

    sync_mode()

    controls = widgets.HBox([modal_mode, card_width, open_btn, close_btn], layout=widgets.Layout(flex_flow="row wrap", gap="8px"))
    ui = widgets.VBox([controls, card, diag], layout=widgets.Layout(width="100%"))
    return {"ui": ui, "slider": slider, "card": card}

slider_modal_lab = build_slider_modal_lab()
display(slider_modal_lab["ui"])



## 10. Stress grid

This grid is useful for quick **visual regression sweeps**. It puts several figure shapes next to each other:

- no sidebar,
- parameters only,
- info only,
- parameters + info,
- crowded sidebar,
- long title / multiple traces.


In [ ]:

def make_stress_card(title, *, params=(), info=False, traces=2):
    fig = make_demo_figure(title, info=info, params=params, traces=traces, plot_height="220px")
    card = widgets.VBox(
        [
            widgets.HTML(f"<b>{title}</b>"),
            fig._layout.root_widget,
        ],
        layout=widgets.Layout(
            border="1px solid rgba(15,23,42,0.10)",
            padding="8px",
            gap="6px",
            width="100%",
        ),
    )
    return card, fig

stress_specs = [
    dict(title="No sidebar", params=(), info=False, traces=1),
    dict(title="Parameters only", params=("a", "b"), info=False, traces=2),
    dict(title="Info only", params=(), info=True, traces=1),
    dict(title="Params + info", params=("a", "b"), info=True, traces=2),
    dict(title="Crowded sidebar (4 params)", params=("a", "b", "c", "d"), info=False, traces=2),
    dict(title="Long title + 4 traces", params=("a",), info=True, traces=4),
]

stress_cards = []
stress_figs = []

for spec in stress_specs:
    card, fig = make_stress_card(**spec)
    stress_cards.append(card)
    stress_figs.append(fig)

stress_grid = widgets.GridBox(
    stress_cards,
    layout=widgets.Layout(
        width="100%",
        grid_template_columns="repeat(auto-fit, minmax(420px, 1fr))",
        gap="12px",
        align_items="start",
    ),
)

display(stress_grid)



## 11. Scratch reproducer

Edit the function below until it matches the **smallest failing case** from your notebook or test. Then use the diagnostics below it to inspect the runtime widget tree and layout traits.


In [ ]:

def build_suspect_case():
    # Replace this body with the exact construction that is misbehaving.
    fig = Figure(default_x_range=(-6, 6), default_y_range=(-3, 3))
    fig._layout.view_stage.layout.height = "320px"
    fig.title = "Replace this with your failing construction"

    fig.plot(a * sin(b * x),x, id="suspect")
    fig.parameter(a, min=-2, max=2, value=1.0, step=0.1)
    fig.parameter(b, min=0.5, max=4.0, value=1.5, step=0.1)

    out = fig.info_manager.get_output("notes", height="90px")
    with out:
        print("Replace this demo with the failing notebook code.")
        print("Then inspect the tree and diagnostics below.")

    with fig:
        print("This is the default Output area.")

    return fig

suspect_fig = build_suspect_case()
display(suspect_fig)


In [ ]:

pd.DataFrame([figure_diag(suspect_fig)])


In [ ]:

show_tree(suspect_fig._layout.root_widget, max_depth=5)



## 12. What to look for when something breaks

When a layout bug shows up, the fastest checks are usually:

1. **Does some ancestor have a real height?**  
   `PlotlyPane` needs one.

2. **Did the sidebar wrap below the plot, or is it still trying to sit beside it?**  
   Inspect `content_wrapper.layout.flex_flow`.

3. **Did a settings/modal widget get parented where you expected?**  
   Inspect `slider.settings_modal in slider.children` versus `slider._modal_host.children`.

4. **Did a hidden container become visible, and did the pane leave `waiting_for_visibility` / `waiting_for_measurement`?**  
   Manual `reflow()` is still useful, but the driver should now retry automatically.

5. **Are you debugging the actual local source tree?**  
   Re-check the source-path table at the top.

This notebook is meant to make all five of those checks cheap.
